# Adaptive Guardrail Layer (AGL) - Exploratory Data Analysis (EDA) Notebook

In [ ]:
import os
import re
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from transformers import AutoTokenizer
from sklearn.feature_extraction.text import CountVectorizer
from wordcloud import WordCloud

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)

INPUT_PATH = OUTPUT_PATH = "../../data/processed/"

input_file = Path(INPUT_PATH) / "dataset_cleaned.csv"
if not os.path.exists(input_file):
    raise FileNotFoundError(f"Could not find input file: {input_file}")

print(f"Input file found: {input_file}")

### Load Cleaned Dataset

In [ ]:
# load dataset
df = pd.read_csv(input_file)

print("Dataset shape:", df.shape)
df.head()

### Dataset Overview

In [ ]:
df.info()

### Check Missing Values

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": df.isna().mean() * 100
})

missing_summary

### Label Distribution

In [ ]:
label_counts = df["label"].value_counts().sort_index()
label_counts

In [ ]:
plt.figure(figsize=(6,4))

sns.countplot(
    data=df,
    x="label"
)

plt.title("Distribution of Labels")
plt.xlabel("Label (0 = Benign, 1 = Malicious)")
plt.ylabel("Count")

plt.show()

### Source Dataset Distribution

In [ ]:
source_counts = df["source_dataset"].value_counts()
source_counts

In [ ]:
plt.figure(figsize=(10,6))

sns.countplot(
    data=df,
    y="source_dataset",
    order=df["source_dataset"].value_counts().index
)

plt.title("Prompt Distribution by Source Dataset")
plt.xlabel("Number of Prompts")
plt.ylabel("Source Dataset")

plt.show()

### Prompt Length Features

In [ ]:
# character length
df["char_length"] = df["prompt"].str.len()
# word count
df["word_count"] = df["prompt"].str.split().str.len()

df[["char_length","word_count"]].describe()

### Character Length Distribution

In [ ]:
plt.figure(figsize=(8,5))

sns.histplot(
    df["char_length"],
    bins=50,
    kde=True
)

plt.title("Distribution of Prompt Character Length")
plt.xlabel("Characters")
plt.ylabel("Frequency")

plt.show()

### Word Count Distribution

In [ ]:
plt.figure(figsize=(8,5))

sns.histplot(
    df["word_count"],
    bins=50,
    kde=True
)

plt.title("Distribution of Prompt Word Count")
plt.xlabel("Words")
plt.ylabel("Frequency")

plt.show()

### Prompt Length by Label

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(
    data=df,
    x="label",
    y="word_count"
)

plt.title("Word Count by Label")
plt.xlabel("Label (0=Benign, 1=Malicious)")
plt.ylabel("Word Count")

plt.show()

### Label Distribution by Dataset Source

In [ ]:
plt.figure(figsize=(10,6))

sns.countplot(
    data=df,
    y="source_dataset",
    hue="label",
    order=df["source_dataset"].value_counts().index
)

plt.title("Label Distribution by Dataset")
plt.xlabel("Count")
plt.ylabel("Dataset Source")

plt.legend(title="Label")

plt.show()

### Token Count per Prompt

In [ ]:
# load tokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# returns number of tokens in a prompt using the tokenizer
def count_tokens(text):
    # tokenize
    tokens = tokenizer.tokenize(str(text))
    # add special tokens used during classification
    return len(tokens) + 2  # CLS + SEP

# compute token counts
df["token_count"] = df["prompt"].apply(count_tokens)
df["token_count"].describe()

### Token Count Distribution

In [ ]:
plt.figure(figsize=(8,5))

sns.histplot(
    df["token_count"],
    bins=50,
    kde=True
)

plt.title("Distribution of Prompt Token Counts")
plt.xlabel("Tokens")
plt.ylabel("Frequency")

plt.show()

### Token Count by Label

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(
    data=df,
    x="label",
    y="token_count"
)

plt.title("Token Count by Label")
plt.xlabel("Label (0 = Benign, 1 = Malicious)")
plt.ylabel("Token Count")

plt.show()

In [ ]:
token_stats = df.groupby("label")["token_count"].describe()
token_stats

### Word Frequency Differences (Malicious vs Benign)

In [ ]:
vectorizer = CountVectorizer(
    stop_words="english",
    max_features=1000
)

X = vectorizer.fit_transform(df["prompt"])

words = vectorizer.get_feature_names_out()

word_counts = np.array(X.sum(axis=0)).flatten()

freq_df = pd.DataFrame({
    "word": words,
    "count": word_counts
}).sort_values("count", ascending=False)

In [ ]:
plt.figure(figsize=(10,6))

sns.barplot(
    data=freq_df.head(20),
    y="word",
    x="count"
)

plt.title("Most Frequent Words in Dataset")
plt.show()

### Word Cloud

In [ ]:
text = " ".join(df[df["label"]==1]["prompt"])

wordcloud = WordCloud(
    width=1000,
    height=400,
    background_color="white"
).generate(text)

plt.imshow(wordcloud)
plt.axis("off")
plt.title("Word Cloud of Malicious Prompts")

plt.show()

### Random Prompt Samples

In [ ]:
print("Random benign prompts:\n")
display(df[df["label"] == 0]["prompt"].sample(10, random_state=42))

In [ ]:
print("\nRandom malicious prompts:\n")
display(df[df["label"] == 1]["prompt"].sample(10, random_state=42))